In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

In [2]:
combined_df = pd.read_excel('Combined_Time_Fix_Data.xlsx')
Dataset_Values = combined_df['Dataset'].unique()
columns = combined_df.columns.values
print(combined_df[columns[3]].unique())
combined_df.drop(columns=columns[0], axis=1, inplace=True) # inplace to modify combined_df instead of copy of object
combined_df.info()
# combined_df[combined_df['Dataset']==Dataset_Values[2]].head()

['Chemotaxis_Diffusion' 'Conjugation_Ra' 'SOS_Spontaneous']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    794 non-null    object 
 1   Time         400 non-null    float64
 2   Dataset      794 non-null    object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  794 non-null    bool   
 5   R_a          794 non-null    bool   
 6   Chemotaxis   794 non-null    bool   
 7   SOS          794 non-null    bool   
dtypes: bool(4), float64(2), object(2)
memory usage: 28.0+ KB


### Data Cleaning

In [3]:
# Removing Outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Function to clean data per dataset
def clean_data_by_dataset(df, dataset_column, target_column):
    cleaned_df_list = []
    unique_datasets = df[dataset_column].unique() # Get unique names from 'Dataset' column.
    
    for dataset in unique_datasets:
        dataset_df = df[df[dataset_column] == dataset]
        cleaned_dataset_df = remove_outliers(dataset_df, target_column) # 'Time' is the target column to remove the outliers from.
        cleaned_df_list.append(cleaned_dataset_df)
    
    cleaned_combined_df = pd.concat(cleaned_df_list)
    cleaned_combined_df.reset_index(drop=True, inplace=True)
    return cleaned_combined_df

In [4]:
# Apply the function
combined_df = clean_data_by_dataset(combined_df, 'Dataset', 'Time')

# Check the result
combined_df[combined_df['Dataset']=='SOS_Spontaneous'].info()
combined_df['Log_Time'] = combined_df['Time'].apply(lambda x: np.log(x))
combined_df

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    0 non-null      object 
 1   Time         0 non-null      float64
 2   Dataset      0 non-null      object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  0 non-null      bool   
 5   R_a          0 non-null      bool   
 6   Chemotaxis   0 non-null      bool   
 7   SOS          0 non-null      bool   
dtypes: bool(4), float64(2), object(2)
memory usage: 0.0+ bytes


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
0,Normal Model,3124.0,Conjugation_Ra,8.046870,True,True,True,True
1,Normal Model,6104.0,Conjugation_Ra,8.716700,True,True,True,True
2,Normal Model,3418.0,Conjugation_Ra,8.136811,True,True,True,True
3,Normal Model,4620.0,Conjugation_Ra,8.438150,True,True,True,True
4,Normal Model,4282.0,Conjugation_Ra,8.362175,True,True,True,True
...,...,...,...,...,...,...,...,...
387,No R_a No Conj,7081.0,Conjugation_Ra,8.865170,False,False,True,True
388,No R_a No Conj,7508.0,Conjugation_Ra,8.923724,False,False,True,True
389,No R_a No Conj,7013.0,Conjugation_Ra,8.855521,False,False,True,True
390,No R_a No Conj,4643.0,Conjugation_Ra,8.443116,False,False,True,True


### Conjugation R_a Regression Analysis

In [9]:
# Filter the dataframe to exclude the 'Chemotaxis_Diffusion' dataset
filtered_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()

# Create the new column with opposite boolean values
filtered_df["No_R_a"] = ~filtered_df["R_a"]

# Add interaction term for R_a * Conjugation
filtered_df['R_a_Conj'] = filtered_df['R_a'] & filtered_df['Conjugation']

# Add interaction term for No_R_a * Conjugation
filtered_df['No_R_a_Conj'] = filtered_df['No_R_a'] & filtered_df['Conjugation']

# Display the first few rows to verify the new column
filtered_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Conjugation', 'R_a', 'R_a_Conj', 'No_R_a_Conj', 'No_R_a']].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,R_a_Conj,No_R_a_Conj,No_R_a
0,Normal Model,3124.0,Conjugation_Ra,8.046870,True,True,True,False,False
1,Normal Model,6104.0,Conjugation_Ra,8.716700,True,True,True,False,False
2,Normal Model,3418.0,Conjugation_Ra,8.136811,True,True,True,False,False
3,Normal Model,4620.0,Conjugation_Ra,8.438150,True,True,True,False,False
4,Normal Model,4282.0,Conjugation_Ra,8.362175,True,True,True,False,False


In [7]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + No_R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.112
Method:                 Least Squares   F-statistic:                     25.65
Date:                Sun, 21 Jul 2024   Prob (F-statistic):           3.42e-11
Time:                        21:08:21   Log-Likelihood:                -200.46
No. Observations:                 392   AIC:                             406.9
Df Residuals:                     389   BIC:                             418.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.5401    

In [12]:
filtered_df['No_R_a_Conj'].info()
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ No_R_a_Conj'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

<class 'pandas.core.series.Series'>
RangeIndex: 392 entries, 0 to 391
Series name: No_R_a_Conj
Non-Null Count  Dtype
--------------  -----
392 non-null    bool 
dtypes: bool(1)
memory usage: 524.0 bytes
                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     46.87
Date:                Sun, 21 Jul 2024   Prob (F-statistic):           2.96e-11
Time:                        21:13:08   Log-Likelihood:                -202.50
No. Observations:                 392   AIC:                             409.0
Df Residuals:                     390   BIC:                             417.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
       

In [11]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ R_a_Conj'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.5848
Date:                Sun, 21 Jul 2024   Prob (F-statistic):              0.445
Time:                        21:11:19   Log-Likelihood:                -224.45
No. Observations:                 392   AIC:                             452.9
Df Residuals:                     390   BIC:                             460.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            8.3592      0.025  

The interaction term R_a_Conj is statistically significant in the simpler model because this model directly captures the combined effect without the added degrees of freedom, larger standard errors, and potential collinearity of separate terms, thereby reducing the t-values and making it harder to achieve statistical significance.

**Conclusion of Analysis and Comparison to Permutation Test:**
- Conjugation combined with a lack of R_a is the most statistically significant predictor, coef = -0.3226 (p < 0.0001).
- R_a being absent is significant, coef = -0.1047 (p = 0.01)
- Conj being present is significant (p < 0.001).
- Significant in the model focusing on interaction, indicating a combined effect that is crucial, the permutation test showed practical significance for the interaction effect.
- Regression results for No_R_a_Conj indicate a significant interaction effect in the model focusing on this term, although the regression model with both conjugation and no R(a) has a larger R-squared indicating a better fit with a variance in log('Time') is 11.2%.

### Chemotaxis Diffusion Regression Analysis

In [36]:
# Filter the dataframe to include only the 'Chemotaxis_Diffusion' dataset
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()

# Display the first few rows to verify
chemotaxis_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Chemotaxis']].head()

,Condition,Time,Dataset,Log_Time,Chemotaxis
0,Chemotaxis,19232,Chemotaxis_Diffusion,9.864331,True
1,Chemotaxis,11287,Chemotaxis_Diffusion,9.331407,True
2,Chemotaxis,21931,Chemotaxis_Diffusion,9.995656,True
3,Chemotaxis,19650,Chemotaxis_Diffusion,9.885833,True
4,Chemotaxis,26696,Chemotaxis_Diffusion,10.192269,True


In [37]:

# Prepare the model formula
formula = 'Log_Time ~ Chemotaxis'

# Fit the OLS regression model
model_chemotaxis = sm.OLS.from_formula(formula, data=chemotaxis_df).fit()

# Display the summary of the model
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     49.35
Date:                Thu, 04 Jul 2024   Prob (F-statistic):           3.52e-11
Time:                        12:51:50   Log-Likelihood:                 38.427
No. Observations:                 196   AIC:                            -72.85
Df Residuals:                     194   BIC:                            -66.30
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.1529      0

### Mixed-Effects Model

In [38]:
# Step 1: Separate Regression Models

# Filter data for Chemotaxis_Diffusion
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()
# Fit OLS regression model for Chemotaxis
model_chemotaxis = sm.OLS.from_formula('Log_Time ~ Chemotaxis', data=chemotaxis_df).fit()
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

# Filter data for Conjugation_Ra
conjugation_ra_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()
# Add interaction term for R_a * Conjugation
conjugation_ra_df['R_a_Conj'] = conjugation_ra_df['R_a'] & conjugation_ra_df['Conjugation']
# Fit OLS regression model for R_a_Conj
model_ra_conj = sm.OLS.from_formula('Log_Time ~ R_a_Conj', data=conjugation_ra_df).fit()
model_ra_conj_summary = model_ra_conj.summary()
print(model_ra_conj_summary)

# Filter data for SOS_Spontaneous
sos_spontaneous_df = combined_df[combined_df['Dataset'] == 'SOS_Spontaneous'].copy()
# Fit OLS regression model for R_a_Conj
model_sos_spontaneous = sm.OLS.from_formula('Log_Time ~ SOS', data=sos_spontaneous_df).fit()
model_sos_spontaneous_summary = model_sos_spontaneous.summary()
print(model_sos_spontaneous_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     49.35
Date:                Thu, 04 Jul 2024   Prob (F-statistic):           3.52e-11
Time:                        12:52:08   Log-Likelihood:                 38.427
No. Observations:                 196   AIC:                            -72.85
Df Residuals:                     194   BIC:                            -66.30
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.1529      0

In [39]:
# Step 2: Mixed-Effects Model
# Combine both datasets for mixed-effects model
combined_df_mixed = pd.concat([chemotaxis_df, conjugation_ra_df, sos_spontaneous_df], ignore_index=True)
combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis']
# Set 'Chemotaxis' values to NaN for the 'Conjugation_Ra' dataset
combined_df_mixed['Chemotaxis'] = combined_df_mixed['Chemotaxis'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
combined_df_mixed['R_a_Conj'] = combined_df_mixed['R_a_Conj'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA


# Verify the changes
combined_df_mixed.info()
combined_df_mixed.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 775 entries, 0 to 774
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    775 non-null    object 
 1   Time         775 non-null    int64  
 2   Dataset      775 non-null    object 
 3   Log_Time     775 non-null    float64
 4   Conjugation  775 non-null    bool   
 5   R_a          775 non-null    bool   
 6   Chemotaxis   775 non-null    bool   
 7   SOS          775 non-null    bool   
 8   R_a_Conj     775 non-null    bool   
dtypes: bool(5), float64(1), int64(1), object(2)
memory usage: 28.1+ KB


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS,R_a_Conj
770,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
771,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
772,Spontaneous,46438,SOS_Spontaneous,10.745873,True,True,True,False,True
773,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
774,Spontaneous,48751,SOS_Spontaneous,10.794481,True,True,True,False,True


In [40]:
# Prepare the mixed-effects model formula
mixed_formula = 'Log_Time ~ Chemotaxis + R_a_Conj + SOS'

# Fit the mixed-effects model
mixed_model = MixedLM.from_formula(mixed_formula, groups='Dataset', data=combined_df_mixed, missing='drop').fit()

# Display the summary of the mixed-effects model
mixed_model_summary = mixed_model.summary()
print(mixed_model_summary)

            Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   Log_Time 
No. Observations:    775       Method:               REML     
No. Groups:          3         Scale:                0.1468   
Min. group size:     194       Log-Likelihood:       -370.7249
Max. group size:     385       Converged:            Yes      
Mean group size:     258.3                                    
--------------------------------------------------------------
                   Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept          10.697    0.489  21.862 0.000  9.738 11.656
Chemotaxis[T.True] -0.202    0.055  -3.685 0.000 -0.309 -0.094
R_a_Conj[T.True]   -0.185    0.045  -4.080 0.000 -0.274 -0.096
SOS[T.True]        -0.832    0.055 -15.119 0.000 -0.940 -0.724
Dataset Var         0.702    1.836                            

